# Experiment Visualization: LLAMEA-VS-GALLAMEA

Visualization of results from experiment **LLAMEA-VS-GALLAMEA_20260412_211856**.

**Methods (5 seeds each):**
- **LLaMEA-Crossover** — LLaMEA with crossover operator
- **GA-LLaMEA** — Full GA-LLaMEA with D-TS bandit operator selection

**Problem:** MA_BBOB (dims=[5], budget_factor=2000, LLM budget=100)

**Run dates:** 2026-04-12 21:18 → 2026-04-13 03:05

**Sections:**
1. Setup and Data Loading
2. Convergence Plots
3. CEG (Code Evolution Graphs)
4. GA-LLaMEA Arm Selection Percentages
5. Boxplots (Fitness Comparison)
6. Fitness Table (Statistical Summary)
7. EAF / ECDF Diagrams
8. Elo Rating (Tournament Ranking)
9. Summary and Comparison

## 1. Setup and Data Loading

In [ ]:
# Core imports
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# BLADE imports
from iohblade.loggers import ExperimentLogger
from iohblade.plots import (
    plot_convergence,
    plot_experiment_CEG,
    plot_boxplot_fitness_hue,
    plot_boxplot_fitness,
    fitness_table,
)

# Plot styling
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_palette('colorblind')

print('All imports successful.')

In [ ]:
# Resolve experiment directory robustly regardless of Jupyter CWD
_EXP_NAME = 'LLAMEA-VS-GALLAMEA_20260412_211856'

def _find_dir(name):
    """Walk up from CWD looking for results/<name>."""
    search = os.path.abspath(os.getcwd())
    for _ in range(6):
        candidate = os.path.join(search, 'results', name)
        if os.path.isdir(candidate):
            return candidate
        search = os.path.dirname(search)
    return None

# Try relative paths first (matches other notebooks), then walk-up search
for _rel in ['../results', 'results', '../../results']:
    _candidate = os.path.join(_rel, _EXP_NAME)
    if os.path.isdir(_candidate):
        EXPERIMENT_DIR = os.path.abspath(_candidate)
        break
else:
    _found = _find_dir(_EXP_NAME)
    if _found:
        EXPERIMENT_DIR = _found
    else:
        raise FileNotFoundError(f'Cannot find results/{_EXP_NAME}. CWD={os.getcwd()}')

print(f'CWD               : {os.getcwd()}')
print(f'Experiment dir    : {EXPERIMENT_DIR}')
print(f'experimentlog.jsonl exists: {os.path.isfile(os.path.join(EXPERIMENT_DIR, "experimentlog.jsonl"))}')

# Load logger using resolved absolute path
logger = ExperimentLogger(EXPERIMENT_DIR, True)

methods, problems = logger.get_methods_problems()
print(f'Methods ({len(methods)}): {sorted(methods)}')
print(f'Problems ({len(problems)}): {problems}')

## 2. Convergence Plots

Comparing the convergence of **LLaMEA-Crossover** vs **GA-LLaMEA** across the LLM evaluation budget.

In [ ]:
# AOCC Convergence
print('AOCC Convergence Plot')
try:
    plot_convergence(logger, metric='AOCC', save=False, budget=100)
    plt.title('AOCC Convergence — LLaMEA-Crossover vs GA-LLaMEA')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error plotting AOCC convergence: {e}')

In [ ]:
# Fitness Convergence
print('Fitness Convergence Plot')
try:
    plot_convergence(logger, metric='Fitness', save=False, budget=100)
    plt.title('Fitness Convergence — LLaMEA-Crossover vs GA-LLaMEA')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error plotting Fitness convergence: {e}')

## 3. CEG (Code Evolution Graphs)

Visualize how code complexity and quality evolve over the optimization budget for each run.

In [ ]:
print('Code Evolution Graphs')
try:
    plot_experiment_CEG(logger, save=False, budget=100, max_seeds=5)
    plt.show()
except Exception as e:
    print(f'Error plotting CEG: {e}')

## 4. GA-LLaMEA Arm Selection Percentages

Visualize how the D-TS bandit distributes operator selections (`mutation`, `crossover`, `random_new`) over the course of each GA-LLaMEA run. Shows whether the bandit learns to prefer certain operators over time.

The `operator` field from the log data is used to compute cumulative selection percentage at each evaluation step.

In [ ]:
# Arm selection percentage over evaluations for GA-LLaMEA runs
try:
    ga_methods = [m for m in methods if 'GA' in m.upper() and 'LLAMEA' in m.upper().replace('-', '')]
    if not ga_methods:
        print('No GA-LLaMEA methods found in the experiment.')
    else:
        for problem in problems:
            data = logger.get_problem_data(problem_name=problem)
            data.replace([-np.inf], 0, inplace=True)
            data.fillna(0, inplace=True)

            for ga_method in ga_methods:
                method_data = data[data['method_name'] == ga_method].copy()
                if method_data.empty:
                    print(f'No data for {ga_method}')
                    continue

                seeds = sorted(method_data['seed'].unique())
                num_seeds = len(seeds)

                fig, axes = plt.subplots(
                    1, num_seeds,
                    figsize=(5 * num_seeds, 4),
                    squeeze=False,
                    sharey=True,
                )

                operator_colors = {
                    'init': '#888888',
                    'mutation': '#1f77b4',
                    'crossover': '#ff7f0e',
                    'random_new': '#2ca02c',
                }
                operator_order = ['init', 'mutation', 'crossover', 'random_new']

                for seed_i, seed in enumerate(seeds):
                    ax = axes[0, seed_i]
                    run_data = method_data[method_data['seed'] == seed].copy().reset_index(drop=True)

                    # Extract operator from top-level field; fall back to metadata
                    operators = []
                    for _, row in run_data.iterrows():
                        op = row.get('operator', None)
                        if op is None or (isinstance(op, float) and np.isnan(op)):
                            meta = row.get('metadata', {})
                            if isinstance(meta, dict):
                                op = meta.get('operator', 'unknown')
                            else:
                                op = 'unknown'
                        operators.append(op)

                    # Compute cumulative percentages
                    eval_indices = list(range(1, len(operators) + 1))
                    cumulative = {op: [] for op in operator_order}
                    counts = {op: 0 for op in operator_order}

                    for i, op in enumerate(operators):
                        if op in counts:
                            counts[op] += 1
                        total = i + 1
                        for op_name in operator_order:
                            cumulative[op_name].append(counts[op_name] / total * 100)

                    # Stacked area plot
                    bottoms = np.zeros(len(eval_indices))
                    for op_name in operator_order:
                        values = np.array(cumulative[op_name])
                        ax.fill_between(
                            eval_indices, bottoms, bottoms + values,
                            alpha=0.7, label=op_name,
                            color=operator_colors.get(op_name, '#999999'),
                        )
                        bottoms += values

                    ax.set_xlim([1, len(operators)])
                    ax.set_ylim([0, 100])
                    ax.set_xlabel('Evaluation')
                    ax.set_title(f'{ga_method} run:{seed}')
                    if seed_i == 0:
                        ax.set_ylabel('Arm Selection %')
                    if seed_i == num_seeds - 1:
                        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)

                fig.suptitle(f'Arm Selection Percentages — {ga_method} ({problem})', fontsize=14, y=1.02)
                plt.tight_layout()
                plt.show()

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f'Error plotting arm percentages: {e}')

## 5. Boxplots (Fitness Comparison)

Distribution of final fitness values found by each method across 5 seeds.

In [ ]:
# Standard boxplot
print('Fitness Boxplot (grouped by method)')
try:
    plot_boxplot_fitness(logger)
    for ax in plt.gcf().get_axes():
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error plotting fitness boxplot: {e}')

In [ ]:
# Boxplot with hue distinction
print('Fitness Boxplot with Hue')
try:
    plot_boxplot_fitness_hue(logger)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error plotting fitness boxplot (hue): {e}')

## 6. Fitness Table (Statistical Summary)

Tabular summary of fitness statistics with mean, standard deviation, and p-values (Wilcoxon test).

In [ ]:
print('Fitness Table (mean +/- std with p-values)')
try:
    table = fitness_table(logger)
    print(table)
except Exception as e:
    print(f'Error generating fitness table: {e}')

## 7. EAF / ECDF Diagrams

EAF (Empirical Attainment Function) and ECDF (Empirical Cumulative Distribution Function) diagrams using IOH-formatted benchmark data (`.dat` files from IOHprofiler).

In [ ]:
# Check for IOH-formatted benchmark data
ioh_data_dirs = []
for subdir in ['ioh-data', 'ioh_data']:
    candidate = os.path.join(EXPERIMENT_DIR, subdir)
    if os.path.isdir(candidate):
        ioh_data_dirs.append(candidate)

ioh_data_available = len(ioh_data_dirs) > 0

if ioh_data_available:
    print(f'IOH data found in {len(ioh_data_dirs)} location(s):')
    for d in ioh_data_dirs:
        print(f'  - {os.path.abspath(d)}')
else:
    print('No IOH-formatted benchmark data found.')
    for subdir in ['ioh-data', 'ioh_data']:
        candidate = os.path.join(EXPERIMENT_DIR, subdir)
        print(f'  - {os.path.abspath(candidate)} (exists: {os.path.isdir(candidate)})')

In [ ]:
# Generate ECDF and EAF plots if IOH data is available
if ioh_data_available:
    try:
        import iohinspector
        import polars as pl

        manager = iohinspector.DataManager()
        for ioh_dir in ioh_data_dirs:
            manager.add_folder(ioh_dir)
        df_ioh = manager.load(monotonic=True, include_meta_data=True)

        print(f'IOH data loaded: {len(df_ioh)} rows')
        print(f'Algorithms: {df_ioh["algorithm_name"].unique().to_list()}')
        print(f'Dimensions: {df_ioh["dimension"].unique().to_list()}')

        # ECDF plot for dimension 5
        fig, ax = plt.subplots(1, 1, figsize=(12, 10))
        _ = iohinspector.plot_ecdf(
            df_ioh.filter(pl.col('dimension') == 5),
            y_max=100, y_min=1e-8, ax=ax, scale_xlog=True
        )
        ax.set_title('ECDF — LLaMEA-Crossover vs GA-LLaMEA (dim=5)')
        plt.tight_layout()
        plt.show()

        # EAF transformation and AOCC
        df_eaf = iohinspector.transform_fval(df_ioh, 1e-8, 1e2)
        aocc = iohinspector.get_aocc(
            df_eaf.filter(pl.col('dimension') == 5),
            10000,
            group_cols=['algorithm_name']
        )
        print('\nAOCC per algorithm (dim=5):')
        print(aocc)

    except ImportError:
        print('iohinspector not installed. Install with: pip install iohinspector')
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'Error generating EAF/ECDF: {e}')
else:
    print('Skipping EAF/ECDF — no IOH data available.')

## 8. Elo Rating (Tournament Ranking)

Elo ratings provide a tournament-based ranking of the algorithms evaluated using IOH benchmark data. The ranking is computed via `iohinspector.get_tournament_ratings` over 100,000 rounds of simulated pairwise tournaments.

**Note:** Requires IOH-formatted data and the `iohinspector` package.

In [ ]:
# Elo Rating (Tournament Ranking)
if ioh_data_available:
    try:
        import iohinspector
        import polars as pl

        # Load IOH data if not already loaded
        if 'df_ioh' not in dir():
            manager = iohinspector.DataManager()
            for ioh_dir in ioh_data_dirs:
                manager.add_folder(ioh_dir)
            df_ioh = manager.load(monotonic=True, include_meta_data=True)

        # Compute Elo ratings
        dt_elo = iohinspector.get_tournament_ratings(df_ioh, nrounds=100000)
        dt_elo['Rating'] = pd.to_numeric(dt_elo['Rating'], errors='coerce')
        dt_elo['Deviation'] = pd.to_numeric(dt_elo['Deviation'], errors='coerce').fillna(0)

        dt_elo_sorted = dt_elo.sort_values(by='algorithm_name').reset_index(drop=True)

        _, ax = plt.subplots(1, 1, figsize=(8, 6))
        sns.pointplot(data=dt_elo_sorted, x='algorithm_name', y='Rating', linestyle='none', ax=ax)

        ax.errorbar(
            dt_elo_sorted['algorithm_name'],
            dt_elo_sorted['Rating'],
            yerr=dt_elo_sorted['Deviation'],
            fmt='o',
            color='blue',
            alpha=0.8,
            capsize=7,
            elinewidth=2.5,
        )
        ax.grid()
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel('')
        ax.set_title('Tournament Ranking — LLaMEA-Crossover vs GA-LLaMEA', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print('\nElo Ratings (sorted by rating, best first):')
        print(dt_elo.sort_values(by='Rating', ascending=False).to_string(index=False))

    except ImportError as e:
        print(f'iohinspector or dependency not installed: {e}')
        print('Install with: pip install iohinspector')
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'Error computing Elo ratings: {e}')
else:
    print('Skipping Elo Rating — no IOH data available.')

## 9. Summary and Comparison

### Methods Overview

| Method | Seeds | Evaluations | Type |
|--------|-------|-------------|------|
| LLaMEA-Crossover | 5 (0–4) | 96/100 | LLaMEA with crossover operator |
| GA-LLaMEA | 5 (0–4) | 100/100 | Full GA with D-TS bandit operator selection |

**Run period:** 2026-04-12 21:18 → 2026-04-13 03:05

### Observations

Refer to the convergence plots, boxplots, fitness table, EAF/ECDF diagrams, and Elo ratings above to draw conclusions about:

1. **Convergence Speed** — Which method converges fastest to high-quality solutions?
2. **Final Fitness** — Which method achieves the best final AOCC scores?
3. **Consistency** — Which method has the lowest variance across seeds?
4. **Operator Preference** — Does the D-TS bandit in GA-LLaMEA learn to favor mutation or crossover over time?
5. **Crossover Impact** — Does adding GA-style operator selection outperform the fixed LLaMEA-Crossover baseline?